### Week 2: Regularized Regression

In Week 1, a linear regression model was developed to predict diabetes status using health indicators from the BRFSS diabetes dataset. The model achieved an R² value of approximately 0.173, indicating that several variables were associated with diabetes status, but much of the variation remained unexplained.

This week explores regularized regression techniques, including Lasso, Ridge, and Elastic Net regression. These methods add penalty terms to the regression objective function to reduce model complexity and improve generalization. Lasso regression can perform feature selection by shrinking some coefficients to zero, Ridge regression reduces coefficient magnitudes while retaining all features, and Elastic Net combines characteristics of both approaches.

The objective of this analysis is to determine whether regularization improves model performance and to identify which health indicators contribute most strongly to diabetes status.

In [6]:
import pandas as pd
import numpy as np

diabetes = pd.read_csv(
    "diabetes_012_health_indicators_BRFSS2015.csv"
)

print(diabetes.shape)

diabetes.head()

(253680, 22)


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [7]:
X = diabetes.drop("Diabetes_012", axis=1)
y = diabetes["Diabetes_012"]

print(X.shape)
print(y.shape)

(253680, 21)
(253680,)


The target variable is Diabetes_012, where higher values indicate progression from no diabetes to prediabetes and diabetes. The remaining health, demographic, and lifestyle variables are used as predictors.

#### Baseline Linear Regression

Before applying regularization techniques, a standard linear regression model was fit to the data. This model serves as a baseline for comparing Ridge, Lasso, and Elastic Net regression models.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

lr = LinearRegression()

lr.fit(X_train, y_train)

lr_preds = lr.predict(X_test)

linear_r2 = r2_score(y_test, lr_preds)
linear_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

print("Linear R²:", linear_r2)
print("Linear RMSE:", linear_rmse)

Linear R²: 0.17330978646059458
Linear RMSE: 0.6322608048598488


#### Baseline Linear Regression Results 
The baseline linear regression model achieved an R^2 value of approximately 0.173 and an RMSE of approximately 0.632. While the model identified meaningful relationships between health indicators and diabetes status, a large portion of the variation remains unexplained. This provides an opportunity to evaluate whether regularization techniques can improve model performance and reduce the impact of less informative predictors.

#### Ridge Regression

Ridge regression is a regularized version of linear regression that adds a penalty based on the squared magnitude of the model coefficients. Unlike ordinary linear regression, Ridge regression discourages large coefficients while retaining all predictor variables in the model.

This approach is particularly useful when predictors may contain overlapping information or when reducing model complexity can improve generalization to unseen data.

In [9]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

ridge.fit(X_train, y_train)

ridge_preds = ridge.predict(X_test)

ridge_r2 = r2_score(y_test, ridge_preds)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_preds))

print("Ridge R²:", ridge_r2)
print("Ridge RMSE:", ridge_rmse)

Ridge R²: 0.17330984065287092
Ridge RMSE: 0.6322607841364555


In [10]:
ridge_coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": ridge.coef_
})

ridge_coef.sort_values(
    by="Coefficient",
    ascending=False
)

,Feature,Coefficient
0,HighBP,0.154032
6,HeartDiseaseorAttack,0.137234
1,HighChol,0.116210
13,GenHlth,0.097865
2,CholCheck,0.095998
16,DiffWalk,0.095365
5,Stroke,0.073865
17,Sex,0.034485
11,AnyHealthcare,0.026258
18,Age,0.016035


#### Ridge Regression Results

The Ridge regression model achieved an R² value of 0.1733 and an RMSE of 0.6323, producing nearly identical performance to the baseline linear regression model. This result was expected because the Week 1 VIF analysis indicated very low multicollinearity among the predictor variables, with all VIF values below 2.

Because the predictors were already relatively independent, Ridge regression had little opportunity to improve model stability or prediction accuracy. The coefficient estimates were slightly reduced in magnitude, but the overall model performance remained essentially unchanged.

### Lasso Regression

Lasso regression applies a penalty based on the absolute values of the coefficients. Unlike Ridge regression, Lasso can shrink coefficients all the way to zero, effectively removing variables from the model. This property allows Lasso to perform feature selection while simultaneously reducing model complexity.

In [11]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.001)

lasso.fit(X_train, y_train)

lasso_preds = lasso.predict(X_test)

lasso_r2 = r2_score(y_test, lasso_preds)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_preds))

print("Lasso R²:", lasso_r2)
print("Lasso RMSE:", lasso_rmse)

lasso_coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lasso.coef_
})

lasso_coef.sort_values(
    by="Coefficient",
    ascending=False
)

Lasso R²: 0.1730960473915888
Lasso RMSE: 0.6323425344483621


,Feature,Coefficient
0,HighBP,0.152239
6,HeartDiseaseorAttack,0.129485
1,HighChol,0.113857
13,GenHlth,0.098908
16,DiffWalk,0.088365
2,CholCheck,0.071819
5,Stroke,0.051304
17,Sex,0.030328
18,Age,0.016946
3,BMI,0.014725


#### Lasso Regression Results

The Lasso regression model achieved an R² value of 0.1731 and an RMSE of 0.6323. Performance was slightly lower than both ordinary linear regression and Ridge regression, although the difference was very small.

Unlike Ridge regression, Lasso performed feature selection by shrinking some coefficients to zero. In this model, the variables Fruits and NoDocbcCost were effectively removed from the model. This suggests that these variables contributed little additional predictive information once the remaining health indicators were included.

Although predictive performance did not improve, Lasso produced a simpler model by eliminating less informative predictors. This tradeoff between model simplicity and predictive performance is one of the primary advantages of Lasso regression.

#### Elastic Net Regression

Elastic Net combines characteristics of both Ridge and Lasso regression. It applies penalties based on both the absolute value and squared value of coefficients, allowing it to perform feature selection while also addressing multicollinearity. This approach can provide a balance between the behaviors of Ridge and Lasso regression.

In [12]:
from sklearn.linear_model import ElasticNet

elastic = ElasticNet(
    alpha=0.001,
    l1_ratio=0.5
)

elastic.fit(X_train, y_train)

elastic_preds = elastic.predict(X_test)

elastic_r2 = r2_score(y_test, elastic_preds)
elastic_rmse = np.sqrt(mean_squared_error(y_test, elastic_preds))

print("Elastic Net R²:", elastic_r2)
print("Elastic Net RMSE:", elastic_rmse)

elastic_coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": elastic.coef_
})

elastic_coef.sort_values(
    by="Coefficient",
    ascending=False
)

Elastic Net R²: 0.17324842207004498
Elastic Net RMSE: 0.6322842704690699


,Feature,Coefficient
0,HighBP,0.152865
6,HeartDiseaseorAttack,0.132673
1,HighChol,0.114895
13,GenHlth,0.098400
16,DiffWalk,0.091445
2,CholCheck,0.082946
5,Stroke,0.061995
17,Sex,0.032476
11,AnyHealthcare,0.017410
18,Age,0.016558


In [13]:
results = pd.DataFrame({
    "Model": ["Linear", "Ridge", "Lasso", "Elastic Net"],
    "R2": [
        linear_r2,
        ridge_r2,
        lasso_r2,
        elastic_r2
    ],
    "RMSE": [
        linear_rmse,
        ridge_rmse,
        lasso_rmse,
        elastic_rmse
    ]
})

results

,Model,R2,RMSE
0,Linear,0.173310,0.632261
1,Ridge,0.173310,0.632261
2,Lasso,0.173096,0.632343
3,Elastic Net,0.173248,0.632284


#### Elastic Net Results

Elastic Net regression achieved an R² value of 0.1732 and an RMSE of 0.6323. Performance was very similar to both Ridge and Lasso regression.

Elastic Net combines the penalties used by Ridge and Lasso regression. As expected, the resulting coefficients were generally smaller than those from ordinary linear regression, while retaining most predictor variables. The PhysHlth variable was reduced to zero, indicating that Elastic Net performed limited feature selection while maintaining characteristics of Ridge regression.

Overall, Elastic Net produced a balance between model simplicity and coefficient stability, although it did not significantly improve predictive performance.

#### Hyperparameter Tuning

In [14]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge

ridge_params = {
    "alpha": [0.001, 0.01, 0.1, 1, 10, 100]
}

ridge_search = GridSearchCV(
    Ridge(),
    ridge_params,
    cv=5,
    scoring="r2"
)

ridge_search.fit(X_train, y_train)

print("Best Parameters:", ridge_search.best_params_)
print("Best CV Score:", ridge_search.best_score_)

Best Parameters: {'alpha': 100}
Best CV Score: 0.17223103644480403


To identify an appropriate regularization strength, Ridge regression was evaluated using GridSearchCV with five-fold cross-validation. Multiple alpha values ranging from 0.001 to 100 were tested.

The best-performing model selected an alpha value of 100 and achieved a cross-validation R² score of approximately 0.172. This result suggests that stronger regularization was slightly preferred during cross-validation. However, the overall model performance remained very similar to the baseline linear regression model.

These findings indicate that while regularization can help control model complexity, the diabetes dataset already exhibited relatively stable predictor relationships and low multicollinearity. As a result, increasing the regularization strength produced only minimal changes in predictive performance.

### Conclusions

This analysis compared ordinary linear regression, Ridge regression, Lasso regression, and Elastic Net regression using the BRFSS diabetes dataset. The objective was to determine whether regularization techniques could improve the prediction of diabetes status and identify the most important health indicators.

The baseline linear regression model achieved an R² value of 0.173 and an RMSE of 0.632. Ridge regression produced nearly identical results, suggesting that multicollinearity was not a significant issue within the dataset. This finding was consistent with the Week 1 VIF analysis, where all predictor variables had relatively low VIF values.

Lasso regression slightly reduced predictive performance but performed feature selection by shrinking the coefficients of Fruits and NoDocbcCost to zero. Elastic Net regression produced results between those of Ridge and Lasso regression, reducing coefficient magnitudes while removing one variable from the model. Although these techniques simplified the model, they did not meaningfully improve predictive accuracy.

Across all models, the strongest predictors of diabetes status remained HighBP, HeartDiseaseorAttack, HighChol, GenHlth, and DiffWalk. These findings suggest that cardiovascular health and overall physical condition are closely associated with diabetes risk. Overall, regularization provided limited benefits because the dataset already exhibited low multicollinearity and relatively stable predictor relationships. Future analyses may benefit more from alternative modeling approaches, such as logistic regression, support vector machines, or tree-based methods, which may be better suited for predicting chronic disease outcomes.